# Ingest & Download — Interactive Notebook

Use this notebook to step through the first two pipeline stages:
1. **Ingest** — fetch track metadata from a SoundCloud playlist (no files downloaded yet)
2. **Download** — pull the actual MP3s

Run each cell in order. Every cell is safe to re-run.

## Cell 1 — Setup

In [ ]:
print("working")

In [ ]:
import sys
from pathlib import Path
import pandas as pd

# Make sure the project root is on the path
ROOT = Path().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Project imports
import config
from database.models import (
    init_db,
    upsert_song,
    update_song_status,
    update_song_duration,
    get_all_songs,
)
from ingest.soundcloud import fetch_playlist, fetch_single
from downloader.download import download_track

# Initialise (creates mashup.db if it doesn't exist)
init_db()
print("Setup complete. DB ready at:", config.DB_PATH)

## Cell 2 — Config check

Confirm the paths are pointing where you expect before doing anything.

In [ ]:
print(f"Database     : {config.DB_PATH}")
print(f"Raw audio    : {config.RAW_DIR}")
print(f"Vocals       : {config.VOCALS_DIR}")
print(f"Instrumentals: {config.INSTRUMENTALS_DIR}")
print(f"yt-dlp format: {config.YTDLP_FORMAT}")

## Cell 3 — Set your URL

Paste a SoundCloud **playlist** URL or a single **track** URL below.

In [ ]:
PLAYLIST_URL = ""  # <-- paste your SoundCloud URL here

if not PLAYLIST_URL:
    raise ValueError("Please set PLAYLIST_URL before continuing.")
print("URL set:", PLAYLIST_URL)

## Cell 4 — Fetch metadata (preview only, nothing saved yet)

Calls yt-dlp to pull track info. For a large playlist this may take a minute.
No files are downloaded and nothing is written to the database yet.

In [ ]:
# Auto-detect single track vs playlist
# SoundCloud single tracks don't contain '/sets/' in the URL
is_single = "/sets/" not in PLAYLIST_URL

if is_single:
    print("Detected single track URL — fetching single track...")
    result = fetch_single(PLAYLIST_URL)
    tracks = [result] if result else []
else:
    print("Detected playlist URL — fetching all tracks...")
    tracks = fetch_playlist(PLAYLIST_URL)

if not tracks:
    print("No tracks returned. Check the URL and that yt-dlp is installed.")
else:
    print(f"Found {len(tracks)} track(s)\n")
    df = pd.DataFrame(tracks)[
        ["title", "artist", "duration_str", "likes", "plays", "genre", "source_url"]
    ]
    pd.set_option("display.max_colwidth", 60)
    pd.set_option("display.max_rows", 50)
    display(df)

## Cell 5 — Write tracks to database

Saves the metadata fetched above. Re-running this cell is safe — existing tracks are updated, not duplicated (keyed on `source_url`).

In [ ]:
if not tracks:
    print("No tracks to insert — run Cell 4 first.")
else:
    inserted_ids = []
    for t in tracks:
        song_id = upsert_song(
            title=t["title"],
            artist=t["artist"],
            source_url=t["source_url"],
            duration_secs=t.get("duration_secs", 0),
            genre=t.get("genre", ""),
            artist_id=t.get("artist_id", ""),
            track_id=t.get("track_id", ""),
            duration_str=t.get("duration_str", ""),
            upload_date=t.get("upload_date", ""),
            likes=t.get("likes", 0),
            reposts=t.get("reposts", 0),
            comments=t.get("comments", 0),
            plays=t.get("plays", 0),
            thumbnail=t.get("thumbnail", ""),
        )
        inserted_ids.append(song_id)

    print(f"Upserted {len(inserted_ids)} track(s) into the database.\n")

    # Show current DB state
    all_songs = get_all_songs()
    db_df = pd.DataFrame(all_songs)[["id", "title", "artist", "status", "duration_str"]]
    display(db_df)

## Cell 6 — Preview queued songs (ready to download)

Shows which songs are in `queued` status and waiting to be downloaded.

In [ ]:
all_songs = get_all_songs()
queued = [s for s in all_songs if s["status"] == "queued"]

if not queued:
    print("No songs queued. Either the DB is empty or all songs are already downloaded.")
else:
    print(f"{len(queued)} song(s) queued for download:\n")
    display(pd.DataFrame(queued)[["id", "title", "artist", "duration_str", "source_url"]])

## Cell 7 — Download

Downloads all queued songs as MP3s into `RAW_DIR`.

- If a file already exists it is skipped (safe to re-run)
- If SoundCloud returns a short preview, the downloader automatically retries via YouTube
- Downloads can take a while — watch the output below for progress

In [ ]:
all_songs = get_all_songs()
queued = [s for s in all_songs if s["status"] == "queued"]

if not queued:
    print("Nothing to download.")
else:
    succeeded, failed = [], []

    for song in queued:
        sid = song["id"]
        label = f"{song['title']} — {song['artist']}"
        print(f"Downloading: {label} ...", end=" ", flush=True)

        result = download_track(
            song_id=sid,
            title=song["title"],
            source_url=song["source_url"],
            artist=song["artist"],
        )

        if result:
            update_song_status(sid, "downloaded", raw_path=str(result.path))
            if result.duration_secs is not None:
                update_song_duration(sid, result.duration_secs)
            print(f"OK  ({result.path.name})")
            succeeded.append(label)
        else:
            update_song_status(sid, "error")
            print("FAILED")
            failed.append(label)

    print(f"\nDone. {len(succeeded)} downloaded, {len(failed)} failed.")
    if failed:
        print("Failed tracks:")
        for f in failed:
            print(" -", f)

## Cell 8 — Results summary

Final view of all songs in the database and a listing of the downloaded files.

In [ ]:
# Database state
print("=== Database ===")
all_songs = get_all_songs()
if all_songs:
    cols = ["id", "title", "artist", "status", "duration_str", "raw_path"]
    available_cols = [c for c in cols if c in all_songs[0]]
    display(pd.DataFrame(all_songs)[available_cols])
else:
    print("No songs in database yet.")

# File listing
print("\n=== Downloaded files ===")
raw_dir = config.RAW_DIR
if raw_dir.exists():
    mp3s = sorted(raw_dir.glob("*.mp3"))
    if mp3s:
        for p in mp3s:
            size_mb = p.stat().st_size / 1_000_000
            print(f"  {p.name}  ({size_mb:.1f} MB)")
    else:
        print(f"No MP3s found in {raw_dir}")
else:
    print(f"Directory does not exist yet: {raw_dir}")